# 0. Imports and Configuration

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import re

In [ ]:
YEAR_COL = "startYear"
GENRE_COL = "genres"
CATEGORY_COL = "movie_category"
RATING_COL = "weightedRating"
WINDOW_SIZE = 3

TOP_GENRES_OVERALL = 5
TOP_GENRES_CATEGORY = 10
TOP_GENRES_RATING = 3

OUTPUT_DIR = "imdb_trend_visuals"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
df = pd.read_csv("complete_movie_data.csv")

# 1. Find top Movies, Actor, and Director

In [ ]:
for rating in ['Indie', 'A', 'AA', 'AAA']:
    eda_df = df[(df['movie_category'] == rating) & ("startYear == 2023")].copy()
    C = eda_df["averageRating"].mean()
    m = eda_df["numVotes"].quantile(0.60)

    eda_df["weightedRating"] = (
        (eda_df["numVotes"] / (eda_df["numVotes"] + m))
        * eda_df["averageRating"]
        +
        (m / (eda_df["numVotes"] + m)) * C
    )
    eda_df = eda_df.sort_values(
        "weightedRating",
        ascending=False
    ).reset_index(drop=True)

    print(f""" {rating} MOVIE:
        {eda_df[["tconst", "primaryTitle", "averageRating", "numVotes", "weightedRating"]].head(1)}
        """
    )
    which_actor = eda_df.query("startYear == 2023").groupby(['actor'], as_index=False).agg(nmoveies = ('tconst', 'count'), wrating = ("weightedRating", "max"))
    print(which_actor.sort_values(['nmoveies', "wrating"],ascending=False).head(1))

    which_direc = eda_df.query("startYear == 2023").groupby(['director'], as_index=False).agg(nmoveies = ('tconst', 'count'), wrating = ("weightedRating", "max"))
    print(which_direc.sort_values(['nmoveies', "wrating"],ascending=False).head(1))
    print('-----------')

# 2. Find Trends

## 2.1 Plot settings and column check

In [ ]:
sns.set_theme(style="whitegrid", context="talk")
plt.rcParams.update({
    "figure.figsize": (16, 9),
    "figure.dpi": 150,
    "savefig.dpi": 200,
    "font.family": "DejaVu Sans",
    "axes.titleweight": "bold",
    "axes.titlesize": 23,
    "axes.labelsize": 15,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 10,
    "legend.title_fontsize": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

required_cols = [YEAR_COL, GENRE_COL, CATEGORY_COL, RATING_COL]
missing_cols = [col for col in required_cols if col not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")


## 2.2 Helper functions

In [ ]:
def split_genres(value):
    if pd.isna(value) or not str(value).strip():
        return []

    value = str(value).strip()
    value = re.sub(r"\bromantic comedy\b", "romantic_comedy", value)
    value = re.sub(r"\bblack comedy\b", "black_comedy", value)
    value = re.sub(r"\bthriller suspense\b", "thriller_suspense", value)

    return [
        genre.replace("_", " ")
        for genre in value.split()
        if genre.strip()
    ] if not re.search(r"\b(?:romantic_comedy|black_comedy|thriller_suspense)\b", value) else [
        genre.replace("_", " ")
        for genre in value.split()
        if genre.strip()
    ]

def save_chart(fig, filename):
    fig.tight_layout()
    fig.savefig(
        os.path.join(OUTPUT_DIR, filename),
        bbox_inches="tight",
        facecolor="white"
    )
    plt.close(fig)


def format_genre_label(label):
    return str(label).replace("_", " ").title()

## 2.3 Final data prep

In [ ]:
year_df = df.dropna(subset=["_plot_year"]).copy()
year_df["_plot_year"] = year_df["_plot_year"].astype(int)
genre_df = year_df[[YEAR_COL, "_plot_year", CATEGORY_COL, RATING_COL, GENRE_COL]].copy()
genre_df["_genre"] = genre_df[GENRE_COL].apply(split_genres)
genre_df = genre_df.explode("_genre")
genre_df = genre_df[
    genre_df["_genre"].notna() & genre_df["_genre"].ne("")
].copy()

# Genre Trends over Time

In [ ]:
genre_counts = (
    genre_df.groupby(["_plot_year", "_genre"])
    .size()
    .unstack(fill_value=0)
    .sort_index()
)

top_genres = genre_counts.sum().nlargest(TOP_GENRES_OVERALL).index
overall_plot = genre_counts[top_genres]

fig, ax = plt.subplots(figsize=(16, 9))

for genre in overall_plot.columns:
    smoothed_data = overall_plot[genre].rolling(window=WINDOW_SIZE, min_periods=1).mean()

    ax.plot(
        overall_plot.index,
        smoothed_data,
        linewidth=2.8,
        marker=None,
        markersize=4,
        label=format_genre_label(genre)
    )

ax.set_title("Movie Genre Trends Over Time", loc="left", pad=18)
ax.set_xlabel("Release Year")
ax.set_ylabel("Number of Movie–Genre Assignments")
ax.legend(
    title="Genre",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    frameon=False
)
ax.grid(axis="y", alpha=0.25)
ax.grid(axis="x", visible=False)

save_chart(fig, "01_genre_trends_overall.png")

# Wtd. Rating Per Genre

In [ ]:
rating_genre_df = genre_df.copy()
rating_genre_df[RATING_COL] = pd.to_numeric(
    rating_genre_df[RATING_COL], errors="coerce"
)
rating_genre_df = rating_genre_df.dropna(
    subset=["_plot_year", RATING_COL, "_genre"]
)

# Select the five most frequent genres across the dataset
top_5_genres = (
    rating_genre_df["_genre"]
    .value_counts()
    .head(5)
    .index
)

# Calculate annual mean weighted rating for each selected genre
rating_trends = (
    rating_genre_df[rating_genre_df["_genre"].isin(top_5_genres)]
    .groupby(["_plot_year", "_genre"])[RATING_COL]
    .mean()
    .unstack()
    .sort_index()
)

fig, ax = plt.subplots(figsize=(16, 9))


for genre in top_5_genres:
    if genre in rating_trends.columns:
        smoothed_data = rating_trends[genre].rolling(window=WINDOW_SIZE, min_periods=1).mean()

        ax.plot(
            rating_trends.index,
            smoothed_data,
            linewidth=2.8,
            marker=None,
            markersize=4,
            label=format_genre_label(genre)
        )

ax.set_title(
    "Weighted-Rating Trends by Genre (Top 5)",
    loc="left",
    pad=18
)
ax.set_xlabel("Release Year")
ax.set_ylabel("Mean Weighted Rating")

ax.legend(
    title="Genre",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    frameon=False
)

ax.grid(axis="y", alpha=0.25)
ax.grid(axis="x", visible=False)

save_chart(fig, "02_weighted_rating_trends_by_genre.png")

# Movie Category Trends over time

In [ ]:
category_year_counts = (
    year_df.groupby(["_plot_year", CATEGORY_COL])
    .size()
    .unstack(fill_value=0)
    .sort_index()
)

fig, ax = plt.subplots(figsize=(16, 9))

for category in category_year_counts.columns:
    smoothed_data = category_year_counts[category].rolling(window=WINDOW_SIZE, min_periods=1).mean()

    ax.plot(
        category_year_counts.index,
        smoothed_data,
        linewidth=2.8,
        marker=None,
        markersize=4,
        label=str(category)
    )

ax.set_title("Movie Production Trends by Category", loc="left", pad=18)
ax.set_xlabel("Release Year")
ax.set_ylabel("Number of Movies")
ax.legend(
    title="Movie Category",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    frameon=False
)
ax.grid(axis="y", alpha=0.25)
ax.grid(axis="x", visible=False)

save_chart(fig, "03_movie_category_trends.png")

# Wtd. Rating per Category

In [ ]:
rating_year_category = (
    year_df.assign(
        **{
            RATING_COL: pd.to_numeric(year_df[RATING_COL], errors="coerce")
        }
    )
    .dropna(subset=[RATING_COL])
    .groupby(["_plot_year", CATEGORY_COL])[RATING_COL]
    .mean()
    .unstack()
    .sort_index()
)

fig, ax = plt.subplots(figsize=(16, 9))

for category in rating_year_category.columns:
    smoothed_data = rating_year_category[category].rolling(window=WINDOW_SIZE, min_periods=1).mean()

    ax.plot(
        rating_year_category.index,
        smoothed_data,
        linewidth=2.8,
        marker=None,
        markersize=4,
        label=str(category)
    )

ax.set_title(
    "Mean Weighted Rating Trends by Movie Category",
    loc="left",
    pad=18
)
ax.set_xlabel("Release Year")
ax.set_ylabel("Mean Weighted Rating")
ax.legend(
    title="Movie Category",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    frameon=False
)
ax.grid(axis="y", alpha=0.25)
ax.grid(axis="x", visible=False)

save_chart(fig, "04_rating_trend_by_category.png")

# Genre per Movie Category

In [ ]:
category_genre_counts = (
    genre_df.groupby([CATEGORY_COL, "_genre"])
    .size()
    .unstack(fill_value=0)
)

# Denominator: number of movies per category, not genre assignments
category_movie_counts = year_df[CATEGORY_COL].value_counts(dropna=True)

category_genre_share = category_genre_counts.div(
    category_movie_counts.reindex(category_genre_counts.index),
    axis=0
) * 100

top_category_genres = (
    category_genre_counts.sum()
    .nlargest(TOP_GENRES_CATEGORY)
    .index
)

heatmap_data = category_genre_share.reindex(
    columns=top_category_genres,
    fill_value=0
)

heatmap_data.columns = [
    format_genre_label(col) for col in heatmap_data.columns
]

fig, ax = plt.subplots(figsize=(16, 8))

sns.heatmap(
    heatmap_data,
    annot=True,
    fmt=".1f",
    cmap="YlGnBu",
    linewidths=0.5,
    cbar_kws={"label": "Movies in Category Containing Genre (%)"},
    ax=ax
)

ax.set_title("Genre Representation by Movie Category", loc="left", pad=18)
ax.set_xlabel("Genre")
ax.set_ylabel("Movie Category")
ax.tick_params(axis="x", rotation=35)
ax.tick_params(axis="y", rotation=0)

save_chart(fig, "05_genre_by_movie_category.png")